# 01 Security universe

Combines the four country lists exported from Datastream Navigator (active and dead equities), assigns a uid to each security and splits the universe into download blocks. Section 3.2.

## Configuration


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

FILES = {
    "DE": os.path.join(DATA_DIR, "German_Market.xlsx"),
    "FR": os.path.join(DATA_DIR, "French_Market.xlsx"),
    "IT": os.path.join(DATA_DIR, "Italian_Market.xlsx"),
    "ES": os.path.join(DATA_DIR, "Spanish_Market.xlsx"),
}

BLOCK_SIZE = 150  # securities per Datastream request
OUT_XLSX = os.path.join(DATA_DIR, "master_universe.xlsx")
OUT_CSV  = os.path.join(DATA_DIR, "master_universe.csv")

## Load the country lists

In [ ]:
frames = []
for country, path in FILES.items():
    df = pd.read_excel(path, sheet_name=0, dtype=str)
    df["country"] = country
    frames.append(df)

master = pd.concat(frames, ignore_index=True)
master.columns = [c.strip() for c in master.columns]
master = master.rename(columns={
    "Start Date": "start_date",
    "Hist.": "hist_year",
    "Full Name": "full_name",
})

master["start_date"] = pd.to_datetime(master["start_date"], format="%m/%d/%Y", errors="coerce")
master["hist_year"]  = pd.to_numeric(master["hist_year"], errors="coerce").astype("Int64")

## Checks

`Symbol` is the key used in the Datastream requests, so it must be present and unique. Securities sharing a name within a country are kept here: duplicate listings are removed after retrieval (Section 3.4.3).

In [ ]:
assert master["Symbol"].notna().all() and master["Symbol"].str.strip().ne("").all()
assert not master["Symbol"].duplicated().any()

same_name = master.duplicated(subset=["full_name", "country"], keep=False)
print("Securities sharing a name within a country:", same_name.sum())

## Identifiers and download blocks

In [ ]:
# mnemonic codes (e.g. D:SAP) vs numeric codes, frequent among delisted securities
master["symbol_type"] = np.where(
    master["Symbol"].astype(str).str.contains(":", na=False), "mnemonic", "numeric")

# RIC is "-" or blank for many delisted securities
master["has_ric"] = (~master["RIC"].isin(["-", ""])) & master["RIC"].notna()

# sort first so that uids and blocks are reproducible
master = master.sort_values(["country", "Symbol"]).reset_index(drop=True)
master.insert(0, "uid",
              [f"{c}{i:04d}" for i, c in zip(range(1, len(master) + 1), master["country"])])

# download blocks: DE-01, DE-02, ...
master["block"] = ""
for c in master["country"].unique():
    idx = master.index[master["country"] == c]
    for pos, i in enumerate(idx):
        master.at[i, "block"] = f"{c}-{(pos // BLOCK_SIZE) + 1:02d}"

COLS = ["uid", "country", "block", "Name", "full_name", "Symbol",
        "symbol_type", "RIC", "has_ric", "start_date", "hist_year",
        "Sector", "Currency", "Exchange", "Market", "Activity"]
master = master[COLS]

In [ ]:
blk = (master.groupby(["block", "country"])
       .agg(n_securities=("uid", "count"),
            n_active=("Activity", lambda s: (s == "Active").sum()),
            n_dead=("Activity", lambda s: (s == "Dead").sum()),
            first_uid=("uid", "first"),
            last_uid=("uid", "last"))
       .reset_index()
       .sort_values("block"))
blk

## Export


In [ ]:
LEGEND = [
    ("uid", "Unique internal identifier (country + progressive). Stable panel key."),
    ("country", "Market country: DE, FR, IT, ES."),
    ("block", "Download block (max 150 securities). Download Datastream series one block at a time."),
    ("Name / full_name", "Short and extended security name as per Navigator."),
    ("Symbol", "Datastream code for requests (mnemonic e.g. D:SAP, or numeric code for older securities)."),
    ("symbol_type", "mnemonic = code with country prefix; numeric = Datastream numeric code (frequent among Dead)."),
    ("RIC / has_ric", "RIC code if present. has_ric=NO is common among delisted: does not affect Datastream download."),
    ("start_date / hist_year", "Start of historical series available in Datastream."),
    ("Sector", "ICB Sector. Useful for sector dummies and possible exclusions (REIT/financials) in analysis."),
    ("Currency", "Series currency (Euro or pre-euro national currency)."),
    ("Exchange / Market", "Domestic trading venue and market of belonging."),
    ("Activity", "Active = listed; Dead = delisted. Both included to avoid survivorship bias."),
    ("", ""),
    ("RE-LISTING NOTE", "Some companies appear twice with different Symbols (Parmalat, Metrovacesa, Telecom Italia): distinct re-listings, each with its own period. Keep both."),
    ("PADDING NOTE", "Dead series show post-delisting padding (last value repeated): to be truncated at the last real variation of the Return Index in the panel."),
]

HEADER_FILL = PatternFill("solid", fgColor="1F3864")
HEADER_FONT = Font(name="Arial", size=10, bold=True, color="FFFFFF")
BASE_FONT   = Font(name="Arial", size=10)

def write_df(ws, df, widths):
    for j, col in enumerate(df.columns, start=1):
        c = ws.cell(row=1, column=j, value=col); c.fill, c.font = HEADER_FILL, HEADER_FONT
    for i, row in enumerate(dataframe_to_rows(df, index=False, header=False), start=2):
        for j, v in enumerate(row, start=1):
            if pd.isna(v):
                v = None
            elif isinstance(v, pd.Timestamp):
                v = v.strftime("%Y-%m-%d")
            ws.cell(row=i, column=j, value=v).font = BASE_FONT
    for j, w in enumerate(widths, start=1):
        ws.column_dimensions[ws.cell(row=1, column=j).column_letter].width = w
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions

wb = Workbook()

ws1 = wb.active; ws1.title = "Universe"
m = master.copy(); m["has_ric"] = m["has_ric"].map({True: "YES", False: "NO"})
write_df(ws1, m, [9,7,7,26,32,10,10,14,7,11,9,30,12,22,9,8])

ws2 = wb.create_sheet("Blocks")
write_df(ws2, blk[["block","country","n_securities","n_active","n_dead","first_uid","last_uid"]],
         [8,8,9,9,8,10,11])

ws3 = wb.create_sheet("Legend")
ws3.cell(row=1, column=1, value="Field").fill = HEADER_FILL
ws3.cell(row=1, column=1).font = HEADER_FONT
ws3.cell(row=1, column=2, value="Description").fill = HEADER_FILL
ws3.cell(row=1, column=2).font = HEADER_FONT
for i, (k, v) in enumerate(LEGEND, start=2):
    ws3.cell(row=i, column=1, value=k).font = Font(name="Arial", size=10, bold=True)
    c = ws3.cell(row=i, column=2, value=v); c.font = BASE_FONT
    c.alignment = Alignment(wrap_text=True, vertical="top")
ws3.column_dimensions["A"].width = 24
ws3.column_dimensions["B"].width = 110

wb.save(OUT_XLSX)
master.to_csv(OUT_CSV, index=False)

print("[SAVED]", OUT_XLSX)
print("[SAVED]", OUT_CSV)

## Output check

In [ ]:
check = pd.read_csv(OUT_CSV, dtype=str)
print("Rows:", len(check), "| Blocks:", check["block"].nunique())
print("\nAllocation Active/Dead per country:")
print(check.groupby(["country","Activity"]).size().unstack(fill_value=0))

# 1. How many securities per country?
print(master.groupby("country").size())

# 2. How many names appear more than once within the same country (potential duplicates)?
dup = master.duplicated(subset=["full_name", "country"], keep=False)
print("\nRows with repeated names in the same country:", dup.sum())
print(master.loc[dup].groupby("country").size())